# Ticket Triage — Unsloth Decision Model (Colab free / T4)

Fine-tune `unsloth/Qwen3.5-2B` into a **decision model** that routes support tickets to one of 12 teams, following https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth

Dataset: `federicoibba/ticket-triage` (`data/decisions/*.parquet`), built by `scripts/to_decision_dataset.py`.
Each row has three JSON columns:
- `state` — ticket text + intake metadata
- `questions` — one 12-option `choice` (which team owns this ticket?)
- `gold` — the owning team

**Before running:** set `COMMIT` below to a commit SHA that contains `data/decisions/`, and pick Runtime → Change runtime type → T4 GPU.

This is the free-tier path: T4, fp16 (no bf16), ~5k training rows.

In [ ]:
# Install/upgrade Unsloth. Re-run every session (free Colab is ephemeral).
!pip install --upgrade --quiet unsloth

# Fail fast if the decision-model API is available on this stack.
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
print('Unsloth decision API OK')

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print('capability', torch.cuda.get_device_capability(0))
print('bf16 supported:', is_bfloat16_supported())

In [ ]:
REPO = 'federicoibba/ticket-triage'
COMMIT = '24638b29ee8f6ad68ce8cc1c5b85ccd8f4ccfc67'  # commit that added data/decisions/
BASE = f'https://raw.githubusercontent.com/{REPO}/{COMMIT}/data/decisions'

from datasets import load_dataset
train = load_dataset('parquet', data_files=f'{BASE}/train_5k.parquet', split='train')
val   = load_dataset('parquet', data_files=f'{BASE}/val.parquet', split='train')
test  = load_dataset('parquet', data_files=f'{BASE}/test.parquet', split='train')
print(len(train), len(val), len(test))
train[0]

In [ ]:
model, tokenizer = FastDecisionModel.from_pretrained(
    model_name='unsloth/Qwen3.5-2B',
    max_seq_length=1024,   # inputs are short; raise if you add long threads
    load_in_4bit=True,
)
model = FastDecisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

In [ ]:
train_items, report = FastDecisionModel.build_dataset(train, tokenizer, model)
print('train:', report)
val_items, val_report = FastDecisionModel.build_dataset(val, tokenizer, model)
print('val:', val_report)
test_items, test_report = FastDecisionModel.build_dataset(test, tokenizer, model)
print('test:', test_report)

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_items,
    eval_dataset=val_items,
    args=TrainingArguments(
        per_device_train_batch_size=2,   # T4 runs Qwen3.5 in float32; lower to 1 if OOM
        gradient_accumulation_steps=16,
        num_train_epochs=2,              # with less data try 3-4
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=10,
        weight_decay=0.01,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        eval_strategy='epoch',
        logging_steps=10,
        save_strategy='no',
        output_dir='outputs',
        report_to='none',
        seed=3407,
    ),
)

In [ ]:
trainer.train()

In [ ]:
metrics = FastDecisionModel.calibrate(model, tokenizer, val_items)
print(metrics)  # held-out accuracy, calibration error (ece), loss

In [ ]:
results = FastDecisionModel.evaluate(model, tokenizer, test_items)
print(results)
# Compare to the TF-IDF baseline on the same temporal test set:
# accuracy 0.908, macro-F1 0.893, top-3 0.984

In [ ]:
import json
FastDecisionModel.for_inference(model)

for i in range(3):
    row = test[i]
    answers = FastDecisionModel.predict(model, tokenizer, row['state'], json.loads(row['questions']))
    print('pred:', answers['team']['answer'], '| gold:', json.loads(row['gold'])['team']['label'])
    print('  probs:', {k: round(v, 3) for k, v in answers['team']['probabilities'].items()})

In [ ]:
model.save_pretrained('qwen-decisions')
model.save_pretrained_merged('qwen-decisions-merged')
print('saved')
# Optional: push to the Hub
# model.push_to_hub('federicoibba/qwen-ticket-router', token='hf_...')

## Troubleshooting / notes

- **Import fails** (`FastDecisionModel` not found): Runtime → Disconnect and delete runtime, then re-run. If it still fails, install from source:
  `!pip install --upgrade --quiet 'unsloth @ git+https://github.com/unslothai/unsloth.git'`
- **`Using float16 precision for qwen3_5 won't work! Using float32`**: expected on a T4 (no bf16). Unsloth runs Qwen3.5 in float32, which uses more VRAM — hence batch size 2. This is not an error.
- **OOM on the T4**: drop `per_device_train_batch_size` to 1 and raise `gradient_accumulation_steps` to 32, or switch to `unsloth/Qwen3.5-0.8B`.
- **Want fp16 (faster, less VRAM)?** Use `unsloth/Llama-3.2-3B-Instruct` in the model cell; it trains in fp16 on a T4 (article benchmark: 79% acc, ~4GB, ~30 min).
- **Free Colab disconnects**: mount Google Drive and point `output_dir` there so checkpoints survive.
- **T4 is fp16-only** (no bf16); the `is_bfloat16_supported()` switches handle this.
- **Options get trimmed** past ~20 described options; we use 12 short ones, so we are safe.
- **Reproducibility**: the dataset is deterministic; LLM training on Colab is not — expect run-to-run variation.
- **Scale up**: swap `train_5k.parquet` for `train.parquet` (14k) once the pipeline works.